# 📊 Clase 2 — Carga y Exploración de un Dataset Financiero Real

## Aplicaciones de Minería de Datos a Economía y Finanzas
### Maestría en Minería de Datos · UTN Facultad Regional Rosario · 2026

**Docente:** Dr. Darío Ezequiel Díaz · drdarioezequieldiaz@gmail.com

**Fecha:** Jueves 16 de abril de 2026

---

### Objetivos del notebook

1. **Cargar** el German Credit Dataset (Statlog) desde OpenML utilizando `scikit-learn`.
2. **Explorar** su estructura dimensional, tipos de datos y primeras observaciones.
3. **Analizar** la distribución de la variable objetivo (riesgo crediticio) y su desbalanceo.
4. **Examinar** las variables categóricas y numéricas con estadísticos descriptivos.
5. **Visualizar** distribuciones, relaciones bivariadas y patrones relevantes.
6. **Diagnosticar** problemas de calidad: valores faltantes, outliers, cardinalidad.
7. **Conectar** cada hallazgo con los conceptos de Database Marketing y CRISP-DM vistos en clase.

### Dataset: German Credit Data (Statlog)

| Característica | Detalle |
|:---|:---|
| **Fuente** | UCI ML Repository / OpenML (ID: credit-g) |
| **Registros** | 1.000 solicitantes de crédito |
| **Variables** | 20 predictoras + 1 objetivo |
| **Variable objetivo** | `class`: good (700) / bad (300) |
| **Desbalanceo** | 70/30 — desbalanceo genuino moderado |
| **Dominio** | Evaluación de riesgo crediticio bancario (Alemania) |

> **Nota:** Este dataset constituye un benchmark clásico en la literatura de scoring crediticio y será nuestro punto de partida para comprender las fases de *Comprensión de los Datos* y *Preparación de los Datos* dentro del marco CRISP-DM.


---
## 1. Configuración del entorno y carga de librerías

Importamos las librerías fundamentales del ecosistema de ciencia de datos en Python. Cada una cumple un rol específico dentro del pipeline analítico.


In [ ]:
# ══════════════════════════════════════════════════════════════
# 1. CONFIGURACIÓN DEL ENTORNO
# ══════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_openml
import warnings

# Configuración estética
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
warnings.filterwarnings('ignore', category=FutureWarning)

# Versiones de las librerías
print("╔══════════════════════════════════════════════════╗")
print("║   Entorno configurado correctamente             ║")
print("╠══════════════════════════════════════════════════╣")
print(f"║   pandas    : {pd.__version__:<37s}║")
print(f"║   numpy     : {np.__version__:<37s}║")
print(f"║   seaborn   : {sns.__version__:<37s}║")
print(f"║   matplotlib: {plt.matplotlib.__version__:<37s}║")
print("╚══════════════════════════════════════════════════╝")


### 📝 Interpretación

Las librerías se han cargado exitosamente. Utilizamos **pandas** para la manipulación tabular, **numpy** para operaciones vectorizadas, **matplotlib** y **seaborn** para visualización estadística, y **scikit-learn** como puente hacia el repositorio OpenML. La configuración estética con `sns.set_theme` garantiza consistencia visual a lo largo de todo el análisis.

---
## 2. Carga del dataset desde OpenML

El German Credit Dataset se encuentra catalogado en OpenML bajo el identificador `credit-g`. La función `fetch_openml` descarga el dataset directamente desde el repositorio, evitando la necesidad de archivos locales — un patrón reproducible y portable ideal para entornos como Google Colab.


In [ ]:
# ══════════════════════════════════════════════════════════════
# 2. CARGA DEL DATASET
# ══════════════════════════════════════════════════════════════

# Descarga desde OpenML (repositorio público, acceso sin autenticación)
credit = fetch_openml('credit-g', version=1, as_frame=True, parser='auto')

# Extraemos el DataFrame completo (features + target ya integrados)
df = credit.frame

print(f"✅ Dataset cargado exitosamente desde OpenML")
print(f"   Nombre: {credit.details['name']}")
print(f"   Dimensiones: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"   Memoria en RAM: {df.memory_usage(deep=True).sum() / 1024:.1f} KB")


### 📝 Interpretación

El dataset se ha descargado correctamente desde OpenML. Contamos con **1.000 observaciones** (solicitantes de crédito) y **21 columnas** (20 variables predictoras + 1 variable objetivo). El volumen en memoria resulta reducido (~60 KB), lo cual permite trabajar con fluidez incluso en entornos con recursos limitados.

Cada fila representa un individuo que solicitó un crédito en un banco alemán, y la variable `class` indica si el crédito resultó **«good»** (cumplió con el pago) o **«bad»** (incumplió). Esta estructura binaria configura un problema de **clasificación supervisada** —el tipo de problema más frecuente en scoring crediticio.

---
## 3. Inspección dimensional y estructural

Antes de cualquier análisis, necesitamos comprender la anatomía del dataset: dimensiones, tipos de datos, nombres de columnas y una muestra de los primeros registros.


In [ ]:
# ══════════════════════════════════════════════════════════════
# 3. INSPECCIÓN DIMENSIONAL Y ESTRUCTURAL
# ══════════════════════════════════════════════════════════════

print("=" * 70)
print("3.1 — PRIMERAS 5 OBSERVACIONES")
print("=" * 70)
df.head()


In [ ]:
# Información estructural completa
print("=" * 70)
print("3.2 — ESTRUCTURA DEL DATAFRAME (df.info())")
print("=" * 70)
df.info()


In [ ]:
# Clasificación de variables por tipo
cat_cols = df.select_dtypes(include=['category', 'object']).columns.tolist()
num_cols = df.select_dtypes(include=['number']).columns.tolist()

print("=" * 70)
print("3.3 — CLASIFICACIÓN DE VARIABLES POR TIPO")
print("=" * 70)
print(f"\n  Variables categóricas: {len(cat_cols)}")
for c in cat_cols:
    n_unique = df[c].nunique()
    print(f"    · {c:<30s} ({n_unique} categorías)")

print(f"\n  Variables numéricas: {len(num_cols)}")
for c in num_cols:
    print(f"    · {c:<30s} (rango: {df[c].min():.0f} – {df[c].max():.0f})")


### 📝 Interpretación

La inspección revela una estructura mixta con **predominio de variables categóricas** (13 de 21), algo habitual en datasets de crédito donde la información del solicitante proviene de formularios con opciones predefinidas (estado civil, tipo de vivienda, propósito del crédito, etc.).

Las **7 variables numéricas** incluyen magnitudes continuas como el monto del crédito (`credit_amount`), la duración en meses (`duration`) y la edad del solicitante (`age`). Estas variables suelen presentar distribuciones asimétricas por la derecha —un patrón recurrente en datos financieros que examinaremos en las visualizaciones.

**Conexión con Database Marketing:** Desde la óptica del marketing analítico, las variables categóricas corresponden a **atributos demográficos y contractuales**, mientras que las numéricas capturan **dimensiones transaccionales y de exposición crediticia**. Ambos tipos resultan esenciales para construir perfiles de clientes segmentables.

---
## 4. Análisis de la variable objetivo: distribución y desbalanceo

La variable `class` constituye el **target** del modelo predictivo. Su distribución condiciona directamente la elección de métricas de evaluación y las técnicas de remuestreo necesarias.


In [ ]:
# ══════════════════════════════════════════════════════════════
# 4. DISTRIBUCIÓN DE LA VARIABLE OBJETIVO
# ══════════════════════════════════════════════════════════════

print("=" * 70)
print("4.1 — DISTRIBUCIÓN DE LA VARIABLE OBJETIVO (class)")
print("=" * 70)

class_counts = df['class'].value_counts()
class_props = df['class'].value_counts(normalize=True) * 100

print("\n  Frecuencias absolutas y relativas:\n")
for label in class_counts.index:
    print(f"    {label:<8s}  →  {class_counts[label]:>4d} observaciones  ({class_props[label]:.1f}%)")

ratio = class_counts.max() / class_counts.min()
print(f"\n  Ratio de desbalanceo (mayoría/minoría): {ratio:.2f}:1")
print(f"  Clase mayoritaria: '{class_counts.idxmax()}' ({class_counts.max()} obs.)")
print(f"  Clase minoritaria: '{class_counts.idxmin()}' ({class_counts.min()} obs.)")


In [ ]:
# Visualización de la distribución de clases
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Gráfico de barras
colors = ['#2a9d8f', '#e76f51']
bars = axes[0].bar(class_counts.index, class_counts.values, color=colors,
                    edgecolor='white', linewidth=1.5, width=0.5)
axes[0].set_title('Distribución de la variable objetivo', fontweight='bold', pad=15)
axes[0].set_xlabel('Clase de riesgo crediticio')
axes[0].set_ylabel('Cantidad de observaciones')

# Etiquetas sobre las barras
for bar, count, prop in zip(bars, class_counts.values, class_props.values):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 10,
                f'{count}\n({prop:.1f}%)', ha='center', va='bottom',
                fontweight='bold', fontsize=11)
axes[0].set_ylim(0, class_counts.max() * 1.2)

# Gráfico de torta
wedges, texts, autotexts = axes[1].pie(
    class_counts.values, labels=class_counts.index,
    autopct='%1.1f%%', colors=colors, startangle=90,
    textprops={'fontsize': 12}, pctdistance=0.6,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
for t in autotexts:
    t.set_fontweight('bold')
axes[1].set_title('Proporción de clases', fontweight='bold', pad=15)

plt.suptitle('German Credit Dataset — Variable objetivo: class',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig_01_distribucion_clases.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n📌 Figura guardada: fig_01_distribucion_clases.png")


### 📝 Interpretación

El dataset exhibe un **desbalanceo de clases genuino de 70/30**: el 70% de los solicitantes fueron clasificados como «good» (cumplieron con el pago) y el 30% como «bad» (incumplieron). Este ratio de aproximadamente **2.33:1** configura un **desbalanceo moderado** —suficiente para distorsionar modelos que optimizan accuracy, pero no tan extremo como los escenarios de fraude (donde la clase minoritaria puede representar menos del 1%).

**Implicancias para el modelado:**
- La **accuracy** no constituye una métrica adecuada: un modelo trivial que prediga siempre «good» alcanzaría 70% de acierto sin aportar valor alguno.
- Métricas como **AUC-ROC**, **F1-score** y el **coeficiente de Gini** serán imprescindibles (Unidad 5).
- Técnicas de **remuestreo** como SMOTE o ADASYN podrían mejorar la sensibilidad hacia la clase minoritaria (Unidad 4).

**Conexión con Database Marketing:** En un contexto bancario real, los «bad» representan el costo directo del riesgo crediticio. Identificarlos correctamente antes del otorgamiento del crédito constituye el objetivo central del **scoring crediticio** —una de las aplicaciones más relevantes del Database Marketing.

---
## 5. Estadísticos descriptivos de las variables numéricas

Examinamos la distribución central, la dispersión y los cuantiles de cada variable numérica para detectar asimetrías, valores extremos y rangos inusuales.


In [ ]:
# ══════════════════════════════════════════════════════════════
# 5. ESTADÍSTICOS DESCRIPTIVOS — VARIABLES NUMÉRICAS
# ══════════════════════════════════════════════════════════════

print("=" * 70)
print("5.1 — RESUMEN ESTADÍSTICO (df.describe())")
print("=" * 70)

desc = df[num_cols].describe().T
desc['IQR'] = desc['75%'] - desc['25%']
desc['CV%'] = (desc['std'] / desc['mean'] * 100).round(1)
desc['skew'] = df[num_cols].skew().round(3)
desc['kurtosis'] = df[num_cols].kurtosis().round(3)

print(desc.to_string())


In [ ]:
# Distribuciones de las variables numéricas con histogramas + KDE
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    if i < 8:
        ax = axes[i]
        sns.histplot(df[col], kde=True, ax=ax, color='#2a9d8f',
                     edgecolor='white', linewidth=0.5, alpha=0.7)
        ax.axvline(df[col].mean(), color='#e76f51', linestyle='--',
                   linewidth=1.5, label=f'Media: {df[col].mean():.1f}')
        ax.axvline(df[col].median(), color='#264653', linestyle='-.',
                   linewidth=1.5, label=f'Mediana: {df[col].median():.1f}')
        ax.set_title(col, fontweight='bold')
        ax.legend(fontsize=7, loc='upper right')

# Ocultar ejes sobrantes
for j in range(len(num_cols), 8):
    axes[j].set_visible(False)

plt.suptitle('Distribuciones de las variables numéricas',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig_02_distribuciones_numericas.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n📌 Figura guardada: fig_02_distribuciones_numericas.png")


### 📝 Interpretación

Los estadísticos descriptivos revelan patrones característicos de los datos financieros:

- **`credit_amount`** (monto del crédito): presenta una **distribución fuertemente asimétrica positiva** (skewness > 1). La media supera a la mediana, indicando que unos pocos créditos de monto muy elevado traccionan el promedio hacia la derecha. Este comportamiento se conoce como *cola pesada por la derecha* y resulta omnipresente en variables monetarias.

- **`duration`** (duración en meses): exhibe picos en valores redondos (12, 24, 36, 48 meses), consistente con los plazos estandarizados que ofrecen las entidades financieras. La distribución multimodal refleja la estructura discreta de los productos crediticios.

- **`age`** (edad del solicitante): se aproxima a una **distribución normal** con media cercana a 35 años. El rango (19–75) confirma que el dataset cubre el espectro completo de la población económicamente activa.

- **`installment_commitment`** (tasa de cuota sobre ingreso): variable discreta con valores 1 a 4, donde 4 indica la mayor carga financiera relativa. Su distribución permite segmentar a los solicitantes por nivel de esfuerzo financiero.

**Conexión con el ciclo RFM:** La variable `duration` funciona como proxy de **Frecuencia** (a mayor plazo, más pagos periódicos), mientras que `credit_amount` captura la dimensión de **Monto** — dos de los tres pilares de la segmentación RFM que estudiamos en la presentación.

---
## 6. Análisis de variables categóricas: frecuencias y distribuciones


In [ ]:
# ══════════════════════════════════════════════════════════════
# 6. ANÁLISIS DE VARIABLES CATEGÓRICAS
# ══════════════════════════════════════════════════════════════

print("=" * 70)
print("6.1 — DISTRIBUCIÓN DE FRECUENCIAS — VARIABLES CATEGÓRICAS")
print("=" * 70)

# Excluimos 'class' del análisis de predictoras
cat_predictors = [c for c in cat_cols if c != 'class']

for col in cat_predictors[:6]:  # Mostramos las primeras 6
    print(f"\n▸ {col}")
    freq = df[col].value_counts()
    prop = df[col].value_counts(normalize=True) * 100
    for cat in freq.index:
        print(f"    {cat:<35s} {freq[cat]:>4d}  ({prop[cat]:>5.1f}%)")


In [ ]:
# Visualización de las 4 variables categóricas más relevantes para scoring
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
key_cats = ['checking_status', 'credit_history', 'purpose', 'employment']

for ax, col in zip(axes.flatten(), key_cats):
    order = df[col].value_counts().index
    sns.countplot(data=df, y=col, order=order, hue='class',
                  palette=['#2a9d8f', '#e76f51'], ax=ax, edgecolor='white')
    ax.set_title(f'Distribución de «{col}» por clase de riesgo',
                 fontweight='bold', fontsize=11)
    ax.set_xlabel('Cantidad')
    ax.set_ylabel('')
    ax.legend(title='Clase', loc='lower right', fontsize=9)

plt.suptitle('Variables categóricas clave — Segmentadas por riesgo crediticio',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig_03_categoricas_clave.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n📌 Figura guardada: fig_03_categoricas_clave.png")


### 📝 Interpretación

El análisis de las variables categóricas arroja hallazgos relevantes para la comprensión del negocio crediticio:

- **`checking_status`** (estado de la cuenta corriente): Los solicitantes **sin cuenta corriente** (`no checking`) representan el grupo más numeroso (~40%), seguidos por quienes mantienen saldo negativo (`<0`, ~27%). La proporción de «bad» resulta sensiblemente mayor entre quienes presentan saldo negativo, lo cual sugiere que esta variable posee **alto poder discriminante** para el scoring.

- **`credit_history`** (historial crediticio): La categoría dominante es `existing paid` (~53%), reflejando clientes con créditos vigentes al día. Los clientes con historial `critical/other existing credit` muestran una proporción inesperadamente baja de defaults —un hallazgo contraintuitivo que puede indicar **selección muestral** o que clientes con más experiencia crediticia gestionan mejor sus obligaciones.

- **`purpose`** (propósito del crédito): `radio/tv` y `new car` lideran las categorías. Los créditos para bienes durables (auto, equipamiento) tienden a mostrar menor tasa de incumplimiento que los destinados a propósitos menos tangibles (educación, vacaciones), posiblemente porque el bien financiado opera como **garantía implícita**.

- **`employment`** (antigüedad laboral): La mayor concentración se observa en el tramo `1<=X<4` años. Los desempleados representan solo el 6%, pero exhiben tasas de default notablemente superiores.

**Conexión con CRISP-DM:** Este análisis corresponde a la fase de **Comprensión de los Datos** (Data Understanding), donde el objetivo no consiste meramente en describir, sino en generar **hipótesis de negocio** que orienten el modelado posterior.

---
## 7. Análisis bivariado: relación entre predictoras y variable objetivo


In [ ]:
# ══════════════════════════════════════════════════════════════
# 7. ANÁLISIS BIVARIADO — NUMÉRICAS vs. CLASE
# ══════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

key_nums = ['credit_amount', 'duration', 'age']
titles = ['Monto del crédito', 'Duración (meses)', 'Edad del solicitante']

for ax, col, title in zip(axes, key_nums, titles):
    sns.boxplot(data=df, x='class', y=col, palette=['#2a9d8f', '#e76f51'],
                ax=ax, width=0.5, flierprops={'marker': 'o', 'markersize': 3})
    ax.set_title(f'{title} por clase de riesgo', fontweight='bold')
    ax.set_xlabel('Clase')
    ax.set_ylabel(col)

    # Agregar medias
    means = df.groupby('class')[col].mean()
    for i, cls in enumerate(means.index):
        ax.scatter(i, means[cls], color='#f4a261', s=80, zorder=5,
                   marker='D', edgecolor='white', linewidth=1.2)
        ax.annotate(f'μ={means[cls]:.0f}', (i, means[cls]),
                    textcoords="offset points", xytext=(15, -5),
                    fontsize=9, color='#f4a261', fontweight='bold')

plt.suptitle('Boxplots — Variables numéricas clave por clase de riesgo (◆ = media)',
             fontsize=13, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig('fig_04_boxplots_bivariado.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n📌 Figura guardada: fig_04_boxplots_bivariado.png")


In [ ]:
# Estadísticos comparativos por clase
print("=" * 70)
print("7.1 — COMPARACIÓN DE MEDIAS POR CLASE DE RIESGO")
print("=" * 70)

comparison = df.groupby('class')[num_cols].agg(['mean', 'median', 'std']).round(2)
for col in ['credit_amount', 'duration', 'age']:
    print(f"\n▸ {col}")
    for cls in ['good', 'bad']:
        m = comparison.loc[cls, (col, 'mean')]
        med = comparison.loc[cls, (col, 'median')]
        s = comparison.loc[cls, (col, 'std')]
        print(f"    {cls:<6s}  →  media={m:>10.2f}   mediana={med:>8.1f}   desvío={s:>10.2f}")


### 📝 Interpretación

El análisis bivariado revela asociaciones significativas entre las variables numéricas y la clase de riesgo:

- **`credit_amount`**: Los clientes «bad» solicitaron, en promedio, **montos más elevados** que los «good». La distribución de los «bad» presenta mayor dispersión y valores extremos más pronunciados. Este hallazgo resulta consistente con la intuición financiera: créditos de mayor cuantía implican mayor exposición al riesgo y, consecuentemente, mayor probabilidad de incumplimiento.

- **`duration`**: Los créditos que resultaron «bad» muestran **plazos más prolongados** en promedio. La duración opera como multiplicador del riesgo: cuanto más extenso el período de repago, mayor la probabilidad de que eventos adversos (desempleo, enfermedad, crisis económica) impacten la capacidad de pago del deudor.

- **`age`**: La diferencia entre clases resulta menos pronunciada, aunque los clientes «bad» tienden a ser **ligeramente más jóvenes**. Solicitantes de menor edad pueden carecer de historial crediticio consolidado y de estabilidad laboral, factores que incrementan el riesgo percibido.

**Insight para scoring crediticio:** Las variables `credit_amount` y `duration` emergen como **candidatas fuertes** para el modelo predictivo. La interacción entre ambas —que podríamos capturar mediante un ratio `credit_amount / duration` (cuota mensual estimada)— probablemente tenga poder predictivo superior al de cada variable individual.

---
## 8. Mapa de correlaciones entre variables numéricas


In [ ]:
# ══════════════════════════════════════════════════════════════
# 8. MATRIZ DE CORRELACIÓN
# ══════════════════════════════════════════════════════════════

# Codificamos la variable objetivo para incluirla en la correlación
df_corr = df[num_cols].copy()
df_corr['target'] = (df['class'] == 'bad').astype(int)

corr_matrix = df_corr.corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.8, linecolor='white',
            cbar_kws={'shrink': 0.8, 'label': 'Coeficiente de Pearson'},
            ax=ax)
ax.set_title('Matriz de correlación — Variables numéricas + target',
             fontweight='bold', fontsize=13, pad=15)
plt.tight_layout()
plt.savefig('fig_05_correlacion.png', dpi=150, bbox_inches='tight')
plt.show()

# Correlaciones con el target
print("\nCorrelación con la variable objetivo (target = 1 si 'bad'):\n")
target_corr = corr_matrix['target'].drop('target').sort_values(key=abs, ascending=False)
for var, corr in target_corr.items():
    direction = "↑ riesgo" if corr > 0 else "↓ riesgo"
    bar = "█" * int(abs(corr) * 30)
    print(f"  {var:<28s} r = {corr:>+.3f}  {bar}  {direction}")

print("\n📌 Figura guardada: fig_05_correlacion.png")


### 📝 Interpretación

La matriz de correlación confirma y cuantifica las observaciones previas:

- **`duration` ↔ target** (r ≈ +0.22): La correlación positiva más fuerte con el riesgo. Mayor duración del crédito se asocia a mayor probabilidad de default, consistente con la lógica financiera de exposición temporal al riesgo.

- **`credit_amount` ↔ target** (r ≈ +0.15): Montos más elevados se correlacionan positivamente con el default, aunque la relación es moderada. Esto sugiere que el monto por sí solo no determina el riesgo; son las **interacciones** con otras variables (ingreso, historial, garantías) las que configuran el perfil de riesgo real.

- **`credit_amount` ↔ `duration`** (r ≈ +0.62): Correlación fuerte y esperable: créditos de mayor monto requieren plazos más largos para amortizarse. Esta **colinealidad** deberá considerarse al momento de seleccionar variables para modelos lineales (Unidad 3, Regresión Logística).

- **`age` ↔ target** (r ≈ −0.08): Correlación débil negativa. La edad, por sí sola, aporta escaso poder predictivo lineal, aunque su interacción con otras variables (empleo, historial) puede resultar informativa.

**Nota metodológica:** La correlación de Pearson solo captura relaciones **lineales**. Variables categóricas como `checking_status` o `credit_history` —que mostraron asociaciones fuertes en el análisis visual— no quedan reflejadas aquí. Para cuantificar su relación con el target, recurriremos a métricas como el **Information Value (IV)** y el **test chi-cuadrado** en la Unidad 2.

---
## 9. Diagnóstico de calidad de datos


In [ ]:
# ══════════════════════════════════════════════════════════════
# 9. DIAGNÓSTICO DE CALIDAD DE DATOS
# ══════════════════════════════════════════════════════════════

print("=" * 70)
print("9.1 — VALORES FALTANTES (Missing Values)")
print("=" * 70)

missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_report = pd.DataFrame({
    'Faltantes': missing,
    'Porcentaje (%)': missing_pct
})
# Mostrar solo si hay faltantes, o confirmar que no hay
if missing.sum() == 0:
    print("\n  ✅ No se detectaron valores faltantes en ninguna variable.")
    print(f"     Dataset completo: {df.shape[0]:,} filas × {df.shape[1]} columnas, 0 NaN.")
else:
    print(missing_report[missing_report['Faltantes'] > 0])


In [ ]:
# Detección de duplicados
print("=" * 70)
print("9.2 — REGISTROS DUPLICADOS")
print("=" * 70)

n_duplicados = df.duplicated().sum()
print(f"\n  Registros duplicados exactos: {n_duplicados}")
if n_duplicados > 0:
    print(f"  ⚠️ Se recomienda investigar los registros duplicados antes del modelado.")
else:
    print(f"  ✅ No se encontraron registros duplicados exactos.")


In [ ]:
# Detección de outliers por IQR en variables numéricas
print("=" * 70)
print("9.3 — DETECCIÓN DE OUTLIERS (Método IQR × 1.5)")
print("=" * 70)

print(f"\n  {'Variable':<28s} {'Outliers':>8s} {'%':>8s} {'Lím. inf.':>10s} {'Lím. sup.':>10s}")
print("  " + "─" * 66)

outlier_summary = {}
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_out = ((df[col] < lower) | (df[col] > upper)).sum()
    pct_out = n_out / len(df) * 100
    outlier_summary[col] = n_out
    flag = "  ⚠" if pct_out > 5 else ""
    print(f"  {col:<28s} {n_out:>8d} {pct_out:>7.1f}% {lower:>10.1f} {upper:>10.1f}{flag}")

print(f"\n  Total de valores atípicos detectados: {sum(outlier_summary.values())}")


In [ ]:
# Cardinalidad de las variables categóricas
print("=" * 70)
print("9.4 — CARDINALIDAD DE VARIABLES CATEGÓRICAS")
print("=" * 70)

print(f"\n  {'Variable':<32s} {'Categorías':>11s} {'Tipo recomendado':>20s}")
print("  " + "─" * 65)

for col in cat_cols:
    n_cat = df[col].nunique()
    if n_cat <= 2:
        rec = "Binaria"
    elif n_cat <= 7:
        rec = "One-Hot Encoding"
    else:
        rec = "Target/Freq Encoding"
    print(f"  {col:<32s} {n_cat:>11d} {rec:>20s}")


### 📝 Interpretación

El diagnóstico de calidad arroja un panorama favorable para este dataset de benchmark:

- **Valores faltantes:** El dataset se encuentra **completo** (0% de missing values), algo infrecuente en datos financieros reales donde la incompletitud alcanza típicamente entre el 5% y el 30% de las variables. En la práctica profesional, la Unidad 2 abordará técnicas de imputación (media, mediana, KNN, MICE) y la clasificación de mecanismos de pérdida según Rubin (1976): MCAR, MAR, MNAR.

- **Duplicados:** No se detectaron registros duplicados exactos, lo cual descarta errores de ingesta o consolidación.

- **Outliers:** La variable `credit_amount` presenta la mayor proporción de valores atípicos por la derecha — consistente con la distribución lognormal que observamos previamente. Estos outliers representan **clientes de alto valor** (high net worth) y no deberían eliminarse sino tratarse con **winsorización** al percentil 99 o transformaciones logarítmicas, preservando la información que contienen.

- **Cardinalidad:** La mayoría de las variables categóricas poseen entre 2 y 5 categorías, lo cual habilita **one-hot encoding** sin explosión dimensional. La variable `purpose` (>8 categorías) y potencialmente `checking_status` podrían requerir **target encoding** o agrupamiento de categorías infrecuentes.

---
## 10. Feature engineering preliminar: variables derivadas con sentido económico


In [ ]:
# ══════════════════════════════════════════════════════════════
# 10. FEATURE ENGINEERING PRELIMINAR
# ══════════════════════════════════════════════════════════════

print("=" * 70)
print("10.1 — CREACIÓN DE VARIABLES DERIVADAS")
print("=" * 70)

# Cuota mensual estimada
df['cuota_mensual_est'] = (df['credit_amount'] / df['duration']).round(2)

# Ratio monto/edad (proxy de capacidad de pago relativa a la etapa de vida)
df['ratio_monto_edad'] = (df['credit_amount'] / df['age']).round(2)

# Categorización de edad en tramos
bins_edad = [18, 25, 35, 45, 55, 100]
labels_edad = ['18-25', '26-35', '36-45', '46-55', '56+']
df['tramo_edad'] = pd.cut(df['age'], bins=bins_edad, labels=labels_edad)

# Flag: cuenta corriente negativa (alto riesgo)
df['checking_negativo'] = (df['checking_status'] == '<0').astype(int)

print("\n  Variables derivadas creadas:")
print(f"    · cuota_mensual_est   = credit_amount / duration")
print(f"    · ratio_monto_edad    = credit_amount / age")
print(f"    · tramo_edad          = age categorizado en 5 tramos")
print(f"    · checking_negativo   = flag binario (checking_status == '<0')")

print(f"\n  Nuevas dimensiones del DataFrame: {df.shape[0]:,} filas × {df.shape[1]} columnas")

# Estadísticos de las nuevas variables
print("\n  Estadísticos de las variables derivadas:\n")
for col in ['cuota_mensual_est', 'ratio_monto_edad']:
    print(f"    {col}:")
    for cls in ['good', 'bad']:
        subset = df[df['class'] == cls][col]
        print(f"      {cls:<6s}  →  media={subset.mean():>8.1f}  mediana={subset.median():>8.1f}  desvío={subset.std():>8.1f}")
    print()


In [ ]:
# Visualización de la cuota mensual estimada por clase
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Cuota mensual estimada
sns.boxplot(data=df, x='class', y='cuota_mensual_est',
            palette=['#2a9d8f', '#e76f51'], ax=axes[0], width=0.4)
axes[0].set_title('Cuota mensual estimada por clase', fontweight='bold')
axes[0].set_ylabel('credit_amount / duration')
axes[0].set_xlabel('Clase de riesgo')

# Tasa de default por tramo de edad
default_by_age = df.groupby('tramo_edad', observed=True)['class'].apply(
    lambda x: (x == 'bad').mean() * 100
).reset_index()
default_by_age.columns = ['Tramo de edad', 'Tasa de default (%)']

bars = axes[1].bar(default_by_age['Tramo de edad'],
                   default_by_age['Tasa de default (%)'],
                   color=['#2a9d8f', '#48baa0', '#7ecfb8', '#f4a261', '#e76f51'],
                   edgecolor='white', linewidth=1.5, width=0.5)
axes[1].set_title('Tasa de default por tramo etario', fontweight='bold')
axes[1].set_ylabel('Tasa de default (%)')
axes[1].set_xlabel('Tramo de edad')
axes[1].axhline(y=30, color='gray', linestyle='--', alpha=0.5, label='Promedio global (30%)')
axes[1].legend(fontsize=9)

# Etiquetas
for bar, val in zip(bars, default_by_age['Tasa de default (%)']):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
                f'{val:.1f}%', ha='center', fontweight='bold', fontsize=10)

plt.suptitle('Feature engineering: variables derivadas con sentido económico',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig_06_feature_engineering.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n📌 Figura guardada: fig_06_feature_engineering.png")


### 📝 Interpretación

El feature engineering preliminar genera variables con **sentido económico-financiero** que trascienden la mera combinación matemática de columnas:

- **`cuota_mensual_est`** (monto / duración): Esta variable proxy de la cuota mensual revela que los clientes «bad» enfrentan cuotas estimadas más elevadas en promedio. Desde la perspectiva del scoring crediticio, esta variable captura la **carga financiera del crédito** de manera más directa que el monto o la duración por separado.

- **Tasa de default por tramo etario:** El análisis segmentado por edad muestra que los solicitantes más jóvenes (18-25 años) exhiben tasas de incumplimiento superiores al promedio. Este patrón refleja menor estabilidad laboral, menor historial crediticio y, potencialmente, menor disciplina financiera en las etapas tempranas de la vida económica activa.

- **`checking_negativo`** (flag binario): Transforma una variable categórica de 4 niveles en un indicador binario que captura la señal de riesgo más fuerte: tener cuenta corriente con saldo negativo.

**Conexión con la Unidad 2:** Estas transformaciones constituyen ejemplos de **feature engineering con sentido económico** —un tema que profundizaremos en la Clase 4. La regla cardinal: las mejores variables derivadas surgen del **conocimiento del dominio financiero**, no de la experimentación computacional ciega.

---
## 11. Resumen ejecutivo y conexión con el proyecto integrador


In [ ]:
# ══════════════════════════════════════════════════════════════
# 11. RESUMEN EJECUTIVO
# ══════════════════════════════════════════════════════════════

print("╔══════════════════════════════════════════════════════════════════╗")
print("║                    RESUMEN EJECUTIVO                           ║")
print("║         German Credit Dataset — Exploración Clase 2            ║")
print("╠══════════════════════════════════════════════════════════════════╣")
print(f"║  Registros totales     :  {df.shape[0]:>6,}                              ║")
print(f"║  Variables originales  :  {21:>6}                              ║")
print(f"║  Variables derivadas   :  {4:>6}                              ║")
print(f"║  Valores faltantes     :  {'0 (dataset completo)':<35s}║")
print(f"║  Duplicados            :  {'0 (sin duplicados)':<35s}║")
print(f"║  Desbalanceo           :  {'70/30 (good/bad) — 2.33:1':<35s}║")
print("╠══════════════════════════════════════════════════════════════════╣")
print("║  HALLAZGOS CLAVE                                              ║")
print("║  ─────────────────────────────────────────────────────────────  ║")
print("║  1. checking_status y credit_history: alto poder discriminante ║")
print("║  2. credit_amount y duration: correlacionadas con default      ║")
print("║  3. Solicitantes jóvenes: mayor tasa de incumplimiento         ║")
print("║  4. Cuota mensual estimada: feature derivada prometedora       ║")
print("║  5. Outliers en credit_amount: clientes high-value, no error   ║")
print("╠══════════════════════════════════════════════════════════════════╣")
print("║  PRÓXIMOS PASOS (Clase 3 — Unidad 2)                         ║")
print("║  ─────────────────────────────────────────────────────────────  ║")
print("║  · EDA en profundidad sobre el dataset Telco Customer Churn    ║")
print("║  · Tratamiento de valores faltantes: MCAR, MAR, MNAR          ║")
print("║  · Detección y tratamiento de outliers con criterio económico  ║")
print("║  · Imputación: media, mediana, KNN, MICE                      ║")
print("╚══════════════════════════════════════════════════════════════════╝")


---
## 📚 Referencias

- **Fayyad, U., Piatetsky-Shapiro, G., & Smyth, P.** (1996). From Data Mining to Knowledge Discovery in Databases. *AI Magazine*, 17(3), 37–54.
- **Dua, D. & Graff, C.** (2019). UCI Machine Learning Repository. University of California, Irvine.
- **Berry, M. J. A. & Linoff, G. S.** (2004). *Data Mining Techniques: For Marketing, Sales, and Customer Relationship Management*. Wiley.
- **Provost, F. & Fawcett, T.** (2013). *Data Science for Business*. O'Reilly Media.
- **Siddiqi, N.** (2012). *Credit Risk Scorecards: Developing and Implementing Intelligent Credit Scoring*. Wiley.

---

> **Dr. Darío Ezequiel Díaz** · drdarioezequieldiaz@gmail.com
>
> Aplicaciones de Minería de Datos a Economía y Finanzas · Maestría en Minería de Datos · UTN Facultad Regional Rosario · 2026
>
> *Notebook correspondiente a la Clase 2 — Jueves 16 de abril de 2026*
